# Load data

In [1]:
import numpy as np
import pandas as pd
from sklearn import set_config
set_config(transform_output="pandas")

data = pd.read_csv("data/train.csv")
X_train = data.copy()
y_train = X_train.pop("addicted_label")
X_train = X_train.drop("id", axis=1)

X_train.head()

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No
1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No
2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes
3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN
4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No


In [2]:
data.corr(numeric_only=True)["addicted_label"]

id                         0.001097
age                        0.004043
daily_screen_time_hours    0.611398
social_media_hours         0.532409
gaming_hours               0.205283
work_study_hours           0.251416
sleep_hours                0.042545
notifications_per_day     -0.011583
app_opens_per_day          0.063482
weekend_screen_time        0.589903
addicted_label             1.000000
Name: addicted_label, dtype: float64

# Feature engineering

In [3]:
from sklearn.pipeline import FunctionTransformer


def add_ratio_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X["social_media_ratio"] = X["social_media_hours"] / (X["daily_screen_time_hours"] + 0.01)
    X["portion_of_day_on_screen"] = X["daily_screen_time_hours"] / (24 - X["sleep_hours"])
    X["unaccounted_screen_time"] = X["daily_screen_time_hours"] - X["social_media_hours"] - X["gaming_hours"] - X["work_study_hours"]
    X["screen_to_work_ratio"] = X["daily_screen_time_hours"] / (X["work_study_hours"] + 0.01)
    return X


def bucket_age(X):
    X = X.copy()
    bins = [0, 17, 20, 23, 26, 29, 32, 100]
    labels = ["-17", "18-20", "21-23", "24-26", "27-29", "30-32", "33+"]
    X["age_bucket"] = pd.cut(X["age"], bins=bins, labels=labels)
    return X.drop(columns=["age"])


ratio_features_transformer = FunctionTransformer(add_ratio_features)
bucket_age_transformer = FunctionTransformer(bucket_age)


# Preprocessing

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer


ordinal_cat = make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder(categories=[["Low", "Medium", "High"]]))
onehot_cat = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(sparse_output=False))
num_mean = make_pipeline(SimpleImputer(strategy="mean"))
num_median = make_pipeline(SimpleImputer(strategy="median"))

ordinal_cat_cols = ["stress_level"]
onehot_cat_cols = ["gender", "academic_work_impact", "age_bucket"]
num_mean_cols = ["daily_screen_time_hours", "sleep_hours", "notifications_per_day", "app_opens_per_day", "weekend_screen_time",
            "portion_of_day_on_screen", "unaccounted_screen_time"]
num_median_cols = ["social_media_hours", "gaming_hours", "work_study_hours", "social_media_ratio", "screen_to_work_ratio"]

col_transform = make_column_transformer(
    (ordinal_cat, ordinal_cat_cols),
    (onehot_cat, onehot_cat_cols),
    (num_mean, num_mean_cols),
    (num_median, num_median_cols),
    remainder="drop"
)

X_train = ratio_features_transformer.fit_transform(X_train)
X_train = bucket_age_transformer.fit_transform(X_train)
X_train = col_transform.fit_transform(X_train)


# Cross validation

In [5]:
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import roc_auc_score
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier
# from catboost import CatBoostClassifier
# from tensorflow import keras

# def build_mlp(input_dim):
#     model = keras.Sequential([
#         keras.layers.Dense(64, activation="relu", input_shape=(input_dim,)),
#         keras.layers.Dense(32, activation="relu"),
#         keras.layers.Dense(1, activation="sigmoid"),
#     ])
#     model.compile(optimizer="adam", loss="binary_crossentropy")
#     return model

# models = {
#     "XGBoost": XGBClassifier(eval_metric="logloss"),
#     "LightGBM": LGBMClassifier(),
#     "CatBoost": CatBoostClassifier(verbose=0),
# }

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scores = {name: [] for name in list(models.keys()) + ["Keras MLP"]}

# for train_idx, val_idx in cv.split(X_train, y_train):
#     X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
#     y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

#     for name, model in models.items():
#         model.fit(X_tr, y_tr)
#         preds = model.predict_proba(X_val)[:, 1]
#         scores[name].append(roc_auc_score(y_val, preds))

#     mlp = build_mlp(X_tr.shape[1])
#     mlp.fit(X_tr, y_tr, epochs=20, batch_size=32, verbose=0)
#     mlp_preds = mlp.predict(X_val, verbose=0).ravel()
#     scores["Keras MLP"].append(roc_auc_score(y_val, mlp_preds))

# for name, score_list in scores.items():
#     print(f"{name}: {np.mean(score_list):.4f} +/- {np.std(score_list):.4f}")

# Hyperparameter tuning

In [ ]:
# import optuna
# from optuna.samplers import TPESampler
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import roc_auc_score
# from catboost import CatBoostClassifier

# N_FOLDS = 2
# N_TRIALS = 20
# RANDOM_STATE = 42

# cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# def objective(trial):
#     params = {
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#         "depth": trial.suggest_int("depth", 4, 8),
#         "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
#         "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 2),
#         "random_strength": trial.suggest_float("random_strength", 0, 2),
#         "border_count": trial.suggest_int("border_count", 32, 128),
#         "iterations": 500,              
#         "eval_metric": "AUC",
#         "verbose": 0,
#         "random_state": RANDOM_STATE,
#         "task_type": "CPU",
#         "thread_count": -1,
#         "early_stopping_rounds": 30,
#     }

#     fold_scores = []
#     for train_idx, val_idx in cv.split(X_train, y_train):
#         X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
#         y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

#         model = CatBoostClassifier(**params)
#         model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

#         preds = model.predict_proba(X_val)[:, 1]
#         fold_scores.append(roc_auc_score(y_val, preds))

#     return np.mean(fold_scores)

# study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=5))
# study.optimize(objective, n_trials=N_TRIALS, timeout=900)

# print("Best ROC-AUC:", study.best_value)
# print("Best params:", study.best_params)

/home/lukasbrookfield/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-22 10:49:09,386] A new study created in memory with name: no-name-2686a4b1-ff0b-485a-a307-039b67134e9c
[I 2026-08-22 10:49:52,877] Trial 0 finished with value: 0.9446461505410366 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 8, 'l2_leaf_reg': 7.587945476302646, 'bagging_temperature': 1.1973169683940732, 'random_strength': 0.31203728088487304, 'border_count': 47}. Best is trial 0 with value: 0.9446461505410366.
[I 2026-08-22 10:50:37,213] Trial 1 finished with value: 0.9428549457328135 and parameters: {'learning_rate': 0.012184186502221764, 'depth': 8, 'l2_leaf_reg': 6.41003510568888, 'bagging_temperature': 1.416145155592091, 'random_strength': 0.041168988591604894, 'border_count': 126}. Best is tria

Best ROC-AUC: 0.9575204731932953
Best params: {'learning_rate': 0.26289012103689435, 'depth': 4, 'l2_leaf_reg': 9.522656887511346, 'bagging_temperature': 1.9847253309206745, 'random_strength': 1.978071872196023, 'border_count': 123}


# Prediction

In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier()

model.fit(X_train, y_train)

X_test = pd.read_csv("data/test.csv")
IDs = X_test.pop("id")

X_test = ratio_features_transformer.transform(X_test)
X_test = bucket_age_transformer.transform(X_test)
X_test = col_transform.transform(X_test)

preds = model.predict_proba(X_test)[:, 1]

sub = pd.DataFrame(data={"addicted_label": preds}, index=IDs)
sub.to_csv("train_new.csv")